In [ ]:
# ----------------- STEP 1: Install These First -----------------
# pip install langchain langchain-huggingface sentence-transformers faiss-cpu transformers pymupdf requests

# ----------------- STEP 2: Load PDF and Split -----------------
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
import os

# Load the PDF
pdf_path = os.path.abspath("../Data/Medical_book.pdf")
if not os.path.isfile(pdf_path):
    raise ValueError(f"❌ PDF not found at: {pdf_path}")

print("📄 Loading and splitting PDF...")
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
text_chunks = text_splitter.split_documents(documents)
print(f"✅ Loaded and split into {len(text_chunks)} chunks.")

# ----------------- STEP 3: Create FAISS Embedding Index -----------------
print("🔍 Creating FAISS index with MiniLM embeddings...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
faiss_index = FAISS.from_documents(text_chunks, embedding=embedding_model)

# Save the FAISS index
index_dir = "medical_index"
faiss_index.save_local(index_dir)
print(f"✅ FAISS index saved at '{index_dir}'.")

# ----------------- STEP 4: Load HF Model -----------------
model_name = "google/flan-t5-base"

print(f"🔄 Loading model: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)
print("✅ Model loaded.")

def call_local_model(prompt):
    response = generator(prompt, max_new_tokens=256, temperature=0.5)[0]
    return response.get("generated_text") or response.get("text", "").strip()

# ----------------- STEP 5: Ask Question -----------------
def ask_question(query):
    docs = faiss_index.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are a helpful medical assistant. Answer ONLY using the context below.

Context:
{context}

Question: {query}
Answer:"""

    return call_local_model(prompt)

# ----------------- STEP 6: Test the Chatbot -----------------
# if __name__ == "__main__":
    # question = "What is Acne?"
    # print("🧠 Asking:", question)
    # answer = ask_question(question)
    # print("💬 Answer:", answer)


c:\Users\Aakash\anaconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Loading and splitting PDF...
✅ Loaded and split into 6600 chunks.
🔍 Creating FAISS index with MiniLM embeddings...
✅ FAISS index saved at 'medical_index'.
🔄 Loading model: google/flan-t5-base...


Device set to use cuda:0


✅ Model loaded.
